### Deploy semantic search using with finetuned model 
The deployment architecture includes: 
- Choose a pretrain BERT model, here we use all-MiniLM-L6-v2 model
- Save the ML models in S3 bucket
- Host the ML models using SageMaker endpoints 
- Create Vector index and load data into the index 
- Create API gateway handels queries from web applications and pass it to lambda 
- Create a Lambda function to call SageMaker endpoints to generate embeddings from user query, and send the query results back to API gateway 
- API gateway sends the search results to frontend, and return search results to the users 

![Semantic_search_finetuned_fullstack](image/Semantic_search_finetune_fullstack.png)

In [13]:
!conda install ipykernel -y

Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [14]:
import torch 
print(torch.__version__)

2.10.0+cu128


In [15]:
#installed in the previous notebook
!pip install -q boto3
!pip install -q requests
!pip install -q requests-aws4auth
!pip install -q opensearch-py
!pip install -q tqdm
!pip install -q install transformers[torch]
!pip install -q transformers==4.52.4
!pip install -q sentence-transformers rank_bm25
!pip install -q sagemaker
!pip install -q datasets

ERROR: Could not find a version that satisfies the requirement install (from versions: none)
ERROR: No matching distribution found for install
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 6.0.1 requires huggingface-hub<2.0.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 6.0.1 requires transformers<6.0.0,>=5.0.0, but you have transformers 4.52.4 which is incompatible.


In [16]:
!pip list | grep transformers

sentence-transformers                6.0.1
transformers                         5.16.1


In [17]:
import sagemaker
role = sagemaker.get_execution_role()
role

2026-09-09 15:26:25,857 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


'arn:aws:iam::759472643633:role/geocore-semantic-search-with-opensearch-stag-NBRole-MnTqEOckqG8y'

In [18]:
import pandas as pd 
from sentence_transformers import SentenceTransformer, util
from datasets import Dataset
import boto3
import torch
import io
import json
import os
import sys
sys.path.append('/home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/src')
sys.path.append('/home/ec2-user/SageMaker/semantic-search-model-evaluation/src/')
sys.path.append('/home/ec2-user/SageMaker/semantic-search-model-evaluation/src/data_processing/')

from tqdm import tqdm

from data_processing.utils.full_processing import process_data_e2e
from src.inference import model_fn, predict_fn

import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [19]:
# Load metadata
def read_parquet_from_s3_as_df(region, s3_bucket, s3_key):
    """
    Load a Parquet file from an S3 bucket into a pandas DataFrame.

    Parameters:
    - region: AWS region where the S3 bucket is located.
    - s3_bucket: Name of the S3 bucket.
    - s3_key: Key (path) to the Parquet file within the S3 bucket.

    Returns:
    - df: pandas DataFrame containing the data from the Parquet file.
    """

    # Setup AWS session and clients
    session = boto3.Session(region_name=region)
    s3 = session.resource('s3')

    # Load the Parquet file as a pandas DataFrame
    object = s3.Object(s3_bucket, s3_key)
    body = object.get()['Body'].read()
    df = pd.read_parquet(io.BytesIO(body))
    return df


# Upload the duplicate date to S3 as a parquet file 
def upload_df_to_s3_as_parquet(df, bucket_name, file_key):
    # Save DataFrame as a Parquet file locally
    parquet_file_path = 'temp.parquet'
    df.to_parquet(parquet_file_path)

    # Create an S3 client
    s3_client = boto3.client('s3')

    # Upload the Parquet file to S3 bucket
    try:
        response = s3_client.upload_file(parquet_file_path, bucket_name, file_key)
        os.remove(parquet_file_path)
        print(f'Uploading {file_key} to {bucket_name} as parquet file')
        # Delete the local Parquet file
        return True
    except Exception as e:
        print(e)
        return False


## 1.Preprocess the text [No need to run for model deployment]

Note, change the parquet bucket name to'-dev', '-stage', or '-prod' based on the environment you are running the file.

In [7]:
# 1. Load the data
df_parquet = read_parquet_from_s3_as_df('ca-central-1', 'webpresence-geocore-geojson-to-parquet-stage', '1-records.parquet')
df_sentinel1 = read_parquet_from_s3_as_df('ca-central-1', 'webpresence-geocore-geojson-to-parquet-stage', '1-sentinel1.parquet')
# df_sentinel2 = read_parquet_from_s3_as_df('ca-central-1', 'webpresence-geocore-geojson-to-parquet-stage', '2-sentinel1.parquet')
df_rcm = read_parquet_from_s3_as_df('ca-central-1', 'webpresence-geocore-geojson-to-parquet-stage', '1-rcm-ard.parquet')

# df = pd.concat([df_parquet, df_sentinel1, df_sentinel2, df_rcm], ignore_index=True)
df = pd.concat([df_parquet, df_sentinel1, df_rcm], ignore_index=True)
df.shape

2026-08-25 13:59:40,169 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
2026-08-25 13:59:41,156 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
2026-08-25 13:59:41,369 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


(22132, 72)

In [ ]:
# 2. Apply all required preprocessing on the data
df_processed = process_data_e2e(df, region='ca-central-1', keep_eoCollections=True)
df_processed.shape

2026-08-25 13:59:42,108 - INFO - Starting processing job. Applying full preprocessing to entire dataset before text_normalization and train-test split.
2026-08-25 13:59:42,119 - INFO - Selected required columns. Dataset shape: (22132, 24)
2026-08-25 13:59:42,120 - INFO - Starting data normalization
2026-08-25 13:59:42,121 - INFO - Replacing 'Not Available; Indisponible' with None in columns: ['features_properties_date_published_date', 'features_properties_date_created_date']
2026-08-25 13:59:42,130 - INFO - Successfully replaced
2026-08-25 13:59:42,131 - INFO - Updating source system name from 'cgp' or None to 'geo-ca'
2026-08-25 13:59:42,135 - INFO - Successfully updated
2026-08-25 13:59:42,136 - INFO - Converting values to lists for columns: ['features_properties_topicCategory', 'features_properties_keywords_en', 'features_properties_keywords_fr']
2026-08-25 13:59:42,747 - INFO - Successfully converted
2026-08-25 13:59:42,748 - INFO - Extracting unique descriptions for features_prope

In [ ]:
print(type(df_processed['text_seq'].head(6)[2]))
print(df_processed['text_seq'].head(6)[2])

### Test Embed (requires update of inference.py import load)

In [ ]:
# getting subset of corpus for test
df_processed_subset = df_processed.iloc[:1000]

In [16]:
# Step 4: Embedding text 
tqdm.pandas()
model_directory ="/home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/model/gte-multi-ft2-se"
model = model_fn(model_directory) #(model, tokenizer)
model[0].eval()
model[1].model_max_length = 1024

# fixing corrupted position_ids buffer according to: https://huggingface.co/Alibaba-NLP/gte-multilingual-base/discussions/30
# embeddings = model[0].embeddings
# max_pos = embeddings.position_ids.size(0)
# embeddings.register_buffer("position_ids", torch.arange(max_pos), persistent=True)

df_processed_subset['vector'] = df_processed_subset['text_seq'].progress_apply(lambda x: predict_fn({"inputs": x}, model))

  0%|          | 1/1000 [00:00<00:11, 86.42it/s]


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:13                                                                                   │
│                                                                                                  │
│   10 # max_pos = embeddings.position_ids.size(0)                                                 │
│   11 # embeddings.register_buffer("position_ids", torch.arange(max_pos), persistent=True)        │
│   12                                                                                             │
│ ❱ 13 df_processed_subset['vector'] = df_processed_subset['text_seq'].progress_apply(lambda x:    │
│   14                                                                                             │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/std.py:923 in inner      │
│                                                                                                  │
│    920 │   │   │   │   # Apply the provided function (in **kwargs)                               │
│    921 │   │   │   │   # on the df using our wrapper (which provides bar updating)               │
│    922 │   │   │   │   try:                                                                      │
│ ❱  923 │   │   │   │   │   return getattr(df, df_function)(wrapper, **kwargs)                    │
│    924 │   │   │   │   finally:                                                                  │
│    925 │   │   │   │   │   t.close()                                                             │
│    926                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/series.py:4943 in │
│ apply                                                                                            │
│                                                                                                  │
│   4940 │   │   │   by_row=by_row,                                                                │
│   4941 │   │   │   args=args,                                                                    │
│   4942 │   │   │   kwargs=kwargs,                                                                │
│ ❱ 4943 │   │   ).apply()                                                                         │
│   4944 │                                                                                         │
│   4945 │   def _reindex_indexer(                                                                 │
│   4946 │   │   self,                                                                             │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/apply.py:1422 in  │
│ apply                                                                                            │
│                                                                                                  │
│   1419 │   │   │   return self.apply_compat()                                                    │
│   1420 │   │                                                                                     │
│   1421 │   │   # self.func is Callable                                                           │
│ ❱ 1422 │   │   return self.apply_standard()                                                      │
│   1423 │                                                                                         │
│   1424 │   def agg(self):                                                                        │
│   1425 │   │   result = super().agg()                      

In [17]:
vector = df_processed_subset['vector'].head(6) 
# print(vector[0])
print(type(vector[0]))
print(len(vector[0]))
print(df_processed_subset['vector'].shape)

<class 'list'>
768
(1000,)


In [18]:
df_processed_subset.head(4)

,features_properties_id,features_geometry_coordinates,features_properties_title_en,features_properties_title_fr,features_properties_description_en,features_properties_description_fr,features_properties_date_published_date,features_properties_keywords_en,features_properties_keywords_fr,features_properties_options,...,features_properties_mappable,features_properties_geo_theme,features_properties_foundational,features_properties_description_normalized_en,features_properties_description_normalized_fr,text_en,text_fr,text_seq,text_para,vector
0,d3881c4c-650d-4070-bf9b-1e00aabf0a1d,"[[[-143, 39.05], [-47, 39.05], [-47, 85], [-14...",Canadian Hydrographic Service Non-Navigational...,Données bathymétriques non navigationnelles (N...,"**CHS NONNA data has been updated: April 1, 20...",**Les données NONNA du Service hydrographique ...,2018-10-11,"[Bathymetry, Depth, Hydrography]","[Bathymétrie, les profondeurs des fonds marins...","[{""url"": ""https://data.chs-shc.ca/"", ""protocol...",...,true,"[environment, foundational]",true,"*CHS NONNA data has been updated: April 1, 202...",*Les données NONNA du Service hydrographique d...,Canadian Hydrographic Service Non-Navigational...,Données bathymétriques non navigationnelles (N...,Canadian Hydrographic Service Non-Navigational...,Canadian Hydrographic Service Non-Navigational...,"[-0.057738080620765686, 0.045166634023189545, ..."
1,3d282116-e556-400c-9306-ca1a3cada77f,"[[[-141.0027151, 41.7], [-52.6, 41.7], [-52.6,...",National Road Network - NRN - GeoBase Series,Réseau routier national - RRN - Série GéoBase,Notice - Format decommissioning\n\nGML (Geogra...,Avis - Changement aux formats offerts\n\nLes f...,2015,"[Canada, Geographic Infrastructure, NRN, Natio...","[Canada, Infrastructure géographique, RRN, Rés...","[{""url"": ""https://geo.statcan.gc.ca/geo_wa/ser...",...,true,"[administration, foundational]",true,Notice - Format decommissioning\nGML (Geograph...,Avis - Changement aux formats offerts\nLes for...,National Road Network - NRN - GeoBase Series\n...,Réseau routier national - RRN - Série GéoBase\...,National Road Network - NRN - GeoBase Series\n...,National Road Network - NRN - GeoBase Series\n...,"[-0.06598811596632004, 0.053342439234256744, -..."
2,b6567c5c-8339-4055-99fa-63f92114d9e4,"[[[-141.003, 41.6755], [-52.6174, 41.6755], [-...",First Nations Location,Localisation des Premières Nations,The First Nations geographic location dataset ...,Le jeu de données des Premières Nations contie...,2015-05-01,"[First Nation, Band, Aboriginal, Indian and No...","[Première Nation, bande, autochtone, Affaires ...","[{""url"": ""https://data.aadnc-aandc.gc.ca/geoma...",...,true,"[administration, society, foundational]",true,The First Nations geographic location dataset ...,Le jeu de données des Premières Nations contie...,"First Nations Location\nkeywords:First Nation,...",Localisation des Premières Nations\nmots-clés ...,"First Nations Location\nkeywords:First Nation,...",First Nations Location\nLocalisation des Premi...,"[-0.0812927633523941, 0.06198863312602043, -0...."
3,522b07b9-78e2-4819-b736-ad9208eb1067,"[[[-141.003, 41.6755], [-52.6174, 41.6755], [-...",Aboriginal Lands of Canada Legislative Boundaries,Limites législatives des terres autochtones du...,The Aboriginal Lands of Canada Legislative Bou...,Le service web des limites législatives des te...,2017-07-28,"[Canada Lands, Indian reserves, Land managemen...","[Terres du Canada, Réserves indiennes, Gestion...","[{""url"": ""https://proxyinternet.nrcan-rncan.gc...",...,true,"[administration, foundational]",true,The Aboriginal Lands of Canada Legislative Bou...,Le service web des limites législatives des te...,Aboriginal Lands of Canada Legislative Boundar...,Limites législatives des terres autochtones du...,Aboriginal Lands of Canada Legislative Boundar...,Aboriginal Lands of Canada Legislative Boundar...,"[-0.07931932061910629, 0.04605397209525108, -0..."


## 2. Generate embeddings using local model

In [20]:
import os
os.environ['ENV']='stage'

Set model_max_length in tokenizer_config.json manually.
Make sure code/inference.py is added to the gte model files.


**Important**: copy custom inference.py code to model folder under a code/ directory

- !mkdir /home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/model/gte-multi-ft-se/code
- !cp /home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/src/inference.py /home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/model/gte-multi-ft2-se/code/


Copy of Contents of inference.py:

```
import logging
import requests

# import os
# import json
# import io
# import time
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModel
import torch.nn.functional as F

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


def model_fn(model_dir):
    # Load model from HuggingFace Hub
    config = AutoConfig.from_pretrained(
        model_dir,
        trust_remote_code=True,
        local_files_only=True
    )
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=True,
        local_files_only=True
    )
    
    model = AutoModel.from_pretrained(
        model_dir,
        config=config,
        trust_remote_code=True,
        local_files_only=True
    )
    
    return model, tokenizer

def predict_fn(data, model_and_tokenizer):
    # destruct model and tokenizer
    model, tokenizer = model_and_tokenizer

    # Tokenize sentences
    sentences = data.pop("inputs", data)
    encoded_input = tokenizer(sentences, padding=True, truncation=True, max_length = tokenizer.model_max_length, return_tensors='pt')

    # Compute token embeddings
    with torch.no_grad():
        model_output = model(**encoded_input)

    # Perform pooling
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

    # Normalize embeddings
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    # return dictonary, which will be json serializable
    return  sentence_embeddings[0].tolist()

```

In [6]:
# DO NOT RERUN - test
# !python ./src/embed_all_records/run_embedding_job.py \
#     --sagemaker_job_name "semantic-search-nbv3se-e2e-processing" \
#     --input_s3 "webpresence-geocore-geojson-to-parquet-stage" \
#     --output_s3 "webpresence-nlp-data-preprocessing-stage" \
#     --model_name "gte-multi-ft-se" \
#     --run_test \

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
2026-08-25 14:05:15,838 - __main__ - INFO - Kickstarting embedding job
usage: run_embedd

In [10]:
# embed all records [remove wait and logs for it to run in background, takes approx 3h]
!python ./src/embed_all_records/run_embedding_job.py --sagemaker_job_name "semantic-search-nbv3se-e2e-processing" --input_s3 "webpresence-geocore-geojson-to-parquet-stage" --output_s3 "webpresence-nlp-data-preprocessing-stage" --model_name "gte-multi-ft2-se"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
2026-08-25 14:09:50,701 - __main__ - INFO - Kickstarting embedding job
2026-08-25 14:09:

## 3. Deploy model using sagemaker 

Pre-requisite: go to the tokenizer_config.json in the model files and change model_max_length to 512. Can do this using the terminal

In [22]:
import boto3
import re
import time
import sagemaker
from sagemaker import get_execution_role
from sagemaker.huggingface.model import HuggingFaceModel

In [16]:
!cd /home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/model/gte-multi-ft2-se && tar czvf ../gte-multi-ft2-se.tar.gz *
# !cd /home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/model/gte-multi-ft-se && tar czvf ../gte-multi-ft-se.tar.gz *

1_Pooling/
1_Pooling/config.json
1_Pooling/.ipynb_checkpoints/
1_Pooling/.ipynb_checkpoints/config-checkpoint.json
2_Normalize/
README.md
code/
code/.ipynb_checkpoints/
code/.ipynb_checkpoints/inference-checkpoint.py
code/inference.py
config.json
config_sentence_transformers.json
configuration.py
eval/
eval/Information-Retrieval_evaluation_results.csv
model.safetensors
modeling.py
modules.json
sentence_bert_config.json
special_tokens_map.json
tokenizer.json
tokenizer_config.json
training_args.bin


In [23]:
LOCAL_MODEL_NAME_TO_DEPLOY =  "gte-multi-ft2-se"
SAGEMAKER_MODEL_NAME = "gte-multi-ft2"
# LOCAL_MODEL_NAME_TO_DEPLOY =  "gte-multi-ft-se"
# SAGEMAKER_MODEL_NAME = "gte-multi-ft-se"


sagemaker_session = sagemaker.Session()
inputs = sagemaker_session.upload_data(path=f'/home/ec2-user/SageMaker/semantic-search-with-amazon-opensearch/model/{LOCAL_MODEL_NAME_TO_DEPLOY}.tar.gz', key_prefix='sentence-transformers-model')
print(f"Response from model upload: {inputs}") 

# Create a SageMaker session and get the execution role to be used later 
role = sagemaker.get_execution_role()

# Deploy with model data 
hub = {
    # 'HF_TASK':'feature-extraction', # auto-builds pipeline with model and tokenizer, returns embeddings
}

# create Hugging Face Model Class
huggingface_model = HuggingFaceModel(
   model_data=inputs,  # path to your trained SageMaker model
   role=role,                                            # IAM role with permissions to create an endpoint
   transformers_version="4.51",                           # Transformers version used (closest to supported)
   pytorch_version="2.6",                                # PyTorch version used
   py_version='py312',                                    # Python version used (closest to supported)
   env=hub
)

# deploy model to SageMaker Inference
predictor = huggingface_model.deploy(
   initial_instance_count=1,
   instance_type="ml.m5.large", # Previous ml.t2.medium is deprecated, t3 is not available within list of valid configs, switching to m5 architecture
   endpoint_name = SAGEMAKER_MODEL_NAME
)


2026-09-09 15:28:57,843 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
Response from model upload: s3://sagemaker-ca-central-1-759472643633/sentence-transformers-model/gte-multi-ft2-se.tar.gz
2026-09-09 15:29:00,626 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
2026-09-09 15:29:00,806 - WARNING - HuggingFaceModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
2026-09-09 15:29:00,822 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: HuggingFaceModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(


2026-09-09 15:29:01,660 - INFO - Creating model with name: huggingface-pytorch-inference-2026-09-09-15-29-01-659
2026-09-09 15:29:02,161 - INFO - Creating endpoint-config with name gte-multi-ft2
2026-09-09 15:29:02,447 - INFO - Creating endpoint with name gte-multi-ft2
------!2026-09-09 15:32:33,382 - WARNING - HuggingFacePredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: HuggingFacePredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(


In [24]:
# example request: you always need to define "inputs"
data = {"inputs":" Today is a sunny and nice day in Ottawa"} 
# request
vector = predictor.predict(data)
len(vector)#{"inputs": ["floods event in Canada", "earthquakes"]}

768

## 4. Create OpenSearch index and load text/vector data into the index 


In [30]:
df = read_parquet_from_s3_as_df('ca-central-1', 'webpresence-nlp-data-preprocessing-stage', 'semantic_search_embeddings-gte-multi-ft2-se.parquet')
df.shape

2026-09-09 18:20:47,879 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


(22132, 34)

In [31]:
# import json
# import time
# import boto3

# from tqdm import tqdm
# from urllib.parse import urlparse
# from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
# import src.opensearch as opensearch
from src.opensearch import get_awsauth_from_secret, create_opensearch_connection, delete_aos_index_if_exists, load_data_to_opensearch_index
# from Preprocess_and_embed_text import read_parquet_from_s3_as_df

In [32]:
df.head(4)

,features_properties_id,features_geometry_coordinates,features_properties_title_en,features_properties_title_fr,features_properties_description_en,features_properties_description_fr,features_properties_date_published_date,features_properties_keywords_en,features_properties_keywords_fr,features_properties_options,...,features_properties_mappable,features_properties_geo_theme,features_properties_foundational,features_properties_description_normalized_en,features_properties_description_normalized_fr,text_en,text_fr,text_seq,text_para,vector
0,eodms-rcm-ard,"[[[-180.0, -90.0], [180.0, -90.0], [180.0, 90....",STAC-Collection - RADARSAT Constellation Missi...,STAC-Collection - RADARSAT Constellation Missi...,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,<NA>,"[SpatioTemporal Asset Catalog, stac, sar, eo, ...","[SpatioTemporal Asset Catalog, stac, rcm, rcm-...","[{""url"": ""https://eodms-sgdot.nrcan-rncan.gc.c...",...,false,[imagery],false,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,STAC-Collection - RADARSAT Constellation Missi...,STAC-Collection - RADARSAT Constellation Missi...,STAC-Collection - RADARSAT Constellation Missi...,STAC-Collection - RADARSAT Constellation Missi...,"[[-0.05734883248806, 0.05576413497328758, -0.0..."
1,eodms-rcm-ard-00018e6e-c08c-4439-b0ae-263ba4a8...,"[[[-107.98, 50.54], [-105.97, 50.54], [-105.97...",STAC-Item - RCM1-OK3433412-PK3549259-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3549259-2-SC30MCP...,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,2025-04-04T11:28:16.645473+00:00,"[SpatioTemporal Asset Catalog, stac, sar, eo, ...","[SpatioTemporal Asset Catalog, stac, rcm, rcm-...","[{""url"": ""https://eodms-sgdot.nrcan-rncan.gc.c...",...,false,[imagery],false,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,STAC-Item - RCM1-OK3433412-PK3549259-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3549259-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3549259-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3549259-2-SC30MCP...,"[[-0.06447061896324158, 0.05838947743177414, -..."
2,eodms-rcm-ard-000439c3-fa89-413a-8993-50fe8139...,"[[[-98.39, 56.61], [-95.58, 56.61], [-95.58, 5...",STAC-Item - RCM1-OK3556292-PK3699728-1-SC30MCP...,STAC-Item - RCM1-OK3556292-PK3699728-1-SC30MCP...,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,2025-07-15T12:41:56.153210+00:00,"[SpatioTemporal Asset Catalog, stac, sar, eo, ...","[SpatioTemporal Asset Catalog, stac, rcm, rcm-...","[{""url"": ""https://eodms-sgdot.nrcan-rncan.gc.c...",...,false,[imagery],false,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,STAC-Item - RCM1-OK3556292-PK3699728-1-SC30MCP...,STAC-Item - RCM1-OK3556292-PK3699728-1-SC30MCP...,STAC-Item - RCM1-OK3556292-PK3699728-1-SC30MCP...,STAC-Item - RCM1-OK3556292-PK3699728-1-SC30MCP...,"[[-0.058503106236457825, 0.0576583668589592, -..."
3,eodms-rcm-ard-0006529a-374c-47cb-9ab4-978fa858...,"[[[-73.14, 47.7], [-71.14, 47.7], [-71.14, 48....",STAC-Item - RCM1-OK3433412-PK3552250-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3552250-2-SC30MCP...,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,2025-04-07T12:45:24.136872+00:00,"[SpatioTemporal Asset Catalog, stac, sar, eo, ...","[SpatioTemporal Asset Catalog, stac, rcm, rcm-...","[{""url"": ""https://eodms-sgdot.nrcan-rncan.gc.c...",...,false,[imagery],false,The RADARSAT Constellation Mission (RCM) is Ca...,La Mission de la Constellation RADARSAT (MCR) ...,STAC-Item - RCM1-OK3433412-PK3552250-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3552250-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3552250-2-SC30MCP...,STAC-Item - RCM1-OK3433412-PK3552250-2-SC30MCP...,"[[-0.05846245586872101, 0.052547745406627655, ..."


In [33]:
df.columns

Index(['features_properties_id', 'features_geometry_coordinates',
       'features_properties_title_en', 'features_properties_title_fr',
       'features_properties_description_en',
       'features_properties_description_fr',
       'features_properties_date_published_date',
       'features_properties_keywords_en', 'features_properties_keywords_fr',
       'features_properties_options', 'features_properties_contact',
       'features_properties_cited', 'features_properties_topicCategory',
       'features_properties_date_created_date',
       'features_properties_spatialRepresentation', 'features_properties_type',
       'features_properties_graphicOverview', 'features_properties_language',
       'features_popularity', 'features_properties_sourceSystemName',
       'features_properties_eoCollection', 'features_properties_eoFilters',
       'temporalExtent', 'features_properties_org',
       'features_properties_mappable', 'features_properties_geo_theme',
       'features_properties_

In [34]:
df.iloc[0]['features_properties_org']

{'en': 'Natural Resources Canada', 'fr': 'Ressources naturelles Canada'}

In [35]:
df.iloc[0]['text_en']

"STAC-Collection - RADARSAT Constellation Mission, CEOS-ARD\nkeywords:SpatioTemporal Asset Catalog, stac, sar, eo, radar, radarsat, canada, rcm\nThe RADARSAT Constellation Mission (RCM) is Canada's third generation of Earth observation satellites. Launched on June 12, 2019, the three identical satellites work together to provide frequent, reliable radar imagery that supports a wide range of applications, including disaster management, environmental monitoring, maritime surveillance, and resource management.\nAs part of Canada's Open Government initiative, Natural Resources Canada (NRCan) produces a Canada-wide *CEOS Analysis Ready Data (ARD)* product covering the country's landmass at 30 m resolution using Compact Polarization (CP) imagery, updated every 12 days. The RCM CEOS-ARD (POL) product is the world's first polarimetric Analysis Ready Data dataset to receive approval from the Committee on Earth Observation Satellites (CEOS).\nTraditionally, users had to search for, order, downlo

In [36]:
import numpy as np

df['vector'] = df['vector'].apply(lambda x: x[0] if isinstance(x, np.ndarray) else x) # flatten vector if nested
df['vector'].iloc[0]

array([-0.05734883,  0.05576413, -0.02155791,  0.01239859,  0.02695446,
       -0.03818827,  0.01593043,  0.00220621,  0.06176485, -0.03431602,
       -0.03516543,  0.00375353, -0.05277827, -0.00724729, -0.0374023 ,
        0.05139132,  0.04843679,  0.04758472,  0.04543806,  0.01658751,
        0.03732302,  0.03272959,  0.0065688 , -0.00533454,  0.01294639,
        0.008801  ,  0.00828888, -0.04894459, -0.02695884,  0.01936491,
       -0.0436781 , -0.03902434,  0.00336892,  0.01342508,  0.05073424,
       -0.01233866,  0.02387898,  0.03332675,  0.01149063,  0.00595412,
       -0.03427663, -0.04196415,  0.03295107,  0.03067205, -0.02650969,
        0.02754686, -0.028095  ,  0.04619481, -0.03260451,  0.03651948,
       -0.00128593,  0.00456296,  0.0263576 ,  0.01770289,  0.05754316,
        0.02434408, -0.06959401,  0.02696146,  0.04812455,  0.01382929,
        0.01726251, -0.01297694,  0.01063029, -0.01005436,  0.004951  ,
       -0.00027916,  0.02188816,  0.02602805,  0.09059728,  0.03

Under the cloudformation template 'geocore-semantic-search-with-opensearch-stage; Output tab, find the values for region, aos_host, and os_secret_id

In [37]:
from opensearchpy import OpenSearch, Urllib3AWSV4SignerAuth, Urllib3HttpConnection

host = "search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com"
region = "ca-central-1"
service = "es"

credentials = boto3.Session().get_credentials()

auth = Urllib3AWSV4SignerAuth(
    credentials,
    region,
    service
)

aos_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=auth,
    use_ssl=True,
    verify_certs=True,
    connection_class=Urllib3HttpConnection,
)

print(aos_client.info())

2026-09-09 18:21:14,094 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
2026-09-09 18:21:14,166 - INFO - GET https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/ [status:200 request:0.071s]
{'name': 'c5c720e62d05e7029e56c0bad3996b11', 'cluster_name': '759472643633:semantic-search', 'cluster_uuid': 'zU9eoXc_Tcy5L6s4Chcd7g', 'version': {'number': '7.10.2', 'build_type': 'tar', 'build_hash': 'unknown', 'build_date': '2025-12-02T19:15:52.128251828Z', 'build_snapshot': False, 'lucene_version': '9.12.1', 'minimum_wire_compatibility_version': '7.10.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'The OpenSearch Project: https://opensearch.org/'}


In [38]:
all_indices = aos_client.cat.indices(format='json')
existing_indices = [index['index'] for index in all_indices]
print("Current indexes:", existing_indices)

2026-09-09 18:21:16,385 - INFO - GET https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/_cat/indices?format=json [status:200 request:0.033s]
Current indexes: ['.plugins-ml-model-group', 'geolocation-index-demo', '.ql-datasources', 'minilm-knn-multilingual', '.plugins-ml-task', '.opendistro-reports-definitions', 'gte-multi-ft2', '.opendistro_security', '.opendistro-reports-instances', 'geolocation-analytics-table', 'vcs-audit-logs-v2', 'minilm-knn', 'geolocation-index-v4', '.opensearch-observability', 'nts-index', '.plugins-ml-model', 'index_name_address_range', 'gte-multi-ft-test', 'geolocation-address-range-index', 'id-audit-logs', 'minilm-pretrain-knn', 'geolocation-index-address-range', 'semantic-search-audit-logs', '.plugins-flow-framework-state', 'minilm-knn-2', 'gte-multi-ft', '.kibana_92668751_admin_1', '.plugins-ml-agent', '.kibana_92668751_admin_2', '.plugins-flow-framework-templates', '.kibana_2', 'vcs-audit-logs', '.kibana_1', '.tasks

In [45]:
# aos_client.indices.delete(index="gte-multi-ft-test")

2026-09-10 19:24:26,450 - INFO - DELETE https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft-test [status:200 request:0.315s]


{'acknowledged': True}

In [39]:
#Create an index 
index_name = "gte-multi-ft2"
knn_index = {
    "settings": {
        "index.knn": True, #This enables the k-nearest neighbor (KNN) search capability on the index.
        "index.knn.space_type": "cosinesimil", #cosine similarity 
        "analysis": {
          "analyzer": {
            "default": {
              "type": "standard",
              "stopwords": "_english_"
            }
          }
        }
    },
    "mappings": {
        "properties": {
            "vector": {
                "type": "knn_vector",
                "dimension": 768,
                "store": True,
                "space_type": "cosinesimil" # defining it again for newer versions of Opensearch
            },
            "coordinates":{
              "type": "geo_shape", 
              "store": True 
            }  
        }
    }
}

In [51]:
# Delete index if it exists
delete_aos_index_if_exists(aos_client, index_to_delete=index_name)

2026-08-26 12:34:41,717 - INFO - GET https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/_cat/indices?format=json [status:200 request:0.026s]
Current indexes: ['.plugins-ml-model-group', 'geolocation-index-demo', '.ql-datasources', 'minilm-knn-multilingual', '.plugins-ml-task', '.opendistro-reports-definitions', '.opendistro_security', '.opendistro-reports-instances', 'geolocation-analytics-table', 'vcs-audit-logs-v2', 'minilm-knn', 'geolocation-index-v4', '.opensearch-observability', 'nts-index', '.plugins-ml-model', 'index_name_address_range', 'gte-multi-ft-test', 'geolocation-address-range-index', 'id-audit-logs', 'minilm-pretrain-knn', 'geolocation-index-address-range', 'semantic-search-audit-logs', '.plugins-flow-framework-state', 'minilm-knn-2', 'gte-multi-ft', '.kibana_92668751_admin_1', '.plugins-ml-agent', '.kibana_92668751_admin_2', '.plugins-flow-framework-templates', '.kibana_2', 'vcs-audit-logs', '.kibana_1', '.tasks', '.plugins-ml-c

In [29]:
#Create a index 
aos_client.indices.create(index=index_name,body=knn_index,ignore=400)

2026-09-09 15:50:50,430 - INFO - PUT https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2 [status:200 request:0.816s]


{'acknowledged': True, 'shards_acknowledged': True, 'index': 'gte-multi-ft2'}

In [40]:
#Load data to OpenSearch Index 
load_data_to_opensearch_index(df, aos_client, index_name)

vector has null values: False


Indexing Records:   0%|          | 0/22132 [00:00<?, ?it/s]

2026-09-09 18:21:46,305 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.445s]


Indexing Records:   0%|          | 1/22132 [00:00<2:44:53,  2.24it/s]

2026-09-09 18:21:46,500 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.192s]


...

2026-09-09 18:29:50,327 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.017s]
2026-09-09 18:29:50,346 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.018s]
2026-09-09 18:29:50,365 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.018s]
2026-09-09 18:29:50,382 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.016s]
2026-09-09 18:29:50,403 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.020s]
2026-09-09 18:29:50,420 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.ama

Indexing Records: 100%|█████████▉| 22129/22132 [08:04<00:00, 51.68it/s]

2026-09-09 18:29:50,441 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.019s]
2026-09-09 18:29:50,458 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.016s]
2026-09-09 18:29:50,477 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_doc [status:201 request:0.017s]


Indexing Records: 100%|██████████| 22132/22132 [08:04<00:00, 45.67it/s]

Error retrieving document count: name 'client' is not defined


In [41]:
print("Done")

Done


In [42]:
#Check 
res = aos_client.search(index=index_name, body={"query": {"match_all": {}}})
print(f"Records loaded into the index {index_name} is {res['hits']['total']['value']}.")

2026-09-09 18:29:58,891 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_search [status:200 request:8.221s]
Records loaded into the index gte-multi-ft2 is 10000.


In [43]:
aos_client.indices.get_mapping(index=index_name)[index_name]["mappings"]["properties"]["vector"]

2026-09-09 18:30:28,904 - INFO - GET https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_mapping [status:200 request:0.062s]


{'type': 'knn_vector',
 'store': True,
 'dimension': 768,
 'space_type': 'cosinesimil'}

In [44]:
aos_client.indices.get_settings(index=index_name)[index_name]["settings"]["index"]

2026-09-09 18:30:29,662 - INFO - GET https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft2/_settings [status:200 request:0.022s]


{'replication': {'type': 'DOCUMENT'},
 'number_of_shards': '5',
 'provided_name': 'gte-multi-ft2',
 'knn.space_type': 'cosinesimil',
 'knn': 'true',
 'creation_date': '1788969049698',
 'analysis': {'analyzer': {'default': {'type': 'standard',
    'stopwords': '_english_'}}},
 'number_of_replicas': '1',
 'uuid': 'Vm1vFlR3Q4SuniulGwIg-g',
 'version': {'created': '136407827'}}

## 5. Test the model endpoints and perform search in OpenSearch index 

In [38]:
from sagemaker_fn import invoke_sagemaker_endpoint_ft

2026-08-25 19:28:42,386 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


In [40]:
endpoint_name ='gte-multi-ft2'
payload = {"inputs": "floods event in Canada"}
vector = invoke_sagemaker_endpoint_ft(endpoint_name, payload)
print(len(vector))

Error invoking SageMaker endpoint gte-multi-ft2: An error occurred (ModelError) when calling the InvokeEndpoint operation: Received client error (400) from primary with message "{
  "code": 400,
  "type": "InternalServerException",
  "message": "unable to mmap 1221487872 bytes from file \u003c/opt/ml/model/model.safetensors\u003e: Cannot allocate memory (12)"
}
". See https://ca-central-1.console.aws.amazon.com/cloudwatch/home?region=ca-central-1#logEventViewer:group=/aws/sagemaker/Endpoints/gte-multi-ft2 in account 759472643633 for more information.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 endpoint_name ='gte-multi-ft2'                                                               │
│   2 payload = {"inputs": "floods event in Canada"}                                               │
│   3 vector = invoke_sagemaker_endpoint_ft(endpoint_name, payload)                                │
│ ❱ 4 print(len(vector))                                                                           │
│   5                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: object of type 'NoneType' has no len()

In [13]:
endpoint_name ='gte-multi-ft2-se'
payload = {"inputs": ["floods event in Canada", "earthquakes"]}

runtime_client = boto3.client('runtime.sagemaker', region_name='ca-central-1')  

response = runtime_client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=json.dumps(payload)
        )

print(response)

{'ResponseMetadata': {'RequestId': '2393cb88-b5c6-454b-b7a0-a25adeb4dfdf', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '2393cb88-b5c6-454b-b7a0-a25adeb4dfdf', 'x-amzn-invoked-production-variant': 'AllTraffic', 'date': 'Tue, 21 Jul 2026 17:35:07 GMT', 'content-type': 'application/json', 'content-length': '16172', 'connection': 'keep-alive'}, 'RetryAttempts': 0}, 'ContentType': 'application/json', 'InvokedProductionVariant': 'AllTraffic', 'Body': <botocore.response.StreamingBody object at 0x7f5492328a00>}


In [14]:
json.loads(response['Body'].read().decode())

[-0.06540855020284653,
 0.051884137094020844,
 -0.06542105972766876,
 0.022129319608211517,
 -0.020327437669038773,
 -0.04043056443333626,
 -0.03482047840952873,
 0.009951615706086159,
 0.0008859827066771686,
 -0.025461973622441292,
 -0.027091972529888153,
 -0.025003228336572647,
 -0.02428983710706234,
 -0.004127128981053829,
 -0.05269454047083855,
 -0.005767416208982468,
 0.0174634400755167,
 0.011371737346053123,
 0.05050159990787506,
 0.017035048454999924,
 0.059427861124277115,
 0.05286277085542679,
 -0.014053180813789368,
 -0.01908683031797409,
 0.009503193199634552,
 -0.014232094399631023,
 0.015337239019572735,
 -0.06617682427167892,
 -0.0327041894197464,
 0.010586082004010677,
 -0.05081013962626457,
 -0.05046484246850014,
 0.01070164144039154,
 -0.0027587618678808212,
 0.032590825110673904,
 0.01294077467173338,
 0.01709040440618992,
 -0.006858904380351305,
 0.011488605290651321,
 -0.005110503174364567,
 0.019115351140499115,
 -0.053415440022945404,
 0.016735345125198364,
 0.02

In [41]:
host = "search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com"
region = "ca-central-1"
service = "es"

credentials = boto3.Session().get_credentials()

auth = Urllib3AWSV4SignerAuth(
    credentials,
    region,
    service
)

aos_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=auth,
    use_ssl=True,
    verify_certs=True,
    connection_class=Urllib3HttpConnection,
)

query={
    "size": 20,
    "query": {
        "knn": {
            "vector":{
                "vector":vector,
                "k":20
            }
        }
    }
}

res = aos_client.search(index='gte-multi-ft', size=20, body=query, request_timeout=55)
query_result=[]
for hit in res['hits']['hits']:
    row=[hit['_id'],hit['_score'],hit['_source']['title_en'], hit['_source']['organisation'],hit['_source']['id']]
    query_result.append(row)
query_result_df = pd.DataFrame(data=query_result,columns=["_id","relevancy_score","title","org",'uuid'])
display(query_result_df)

2026-08-25 19:29:39,210 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
2026-08-25 19:29:39,255 - WARNING - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft/_search?size=20 [status:400 request:0.044s]


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:33                                                                                   │
│                                                                                                  │
│   30 │   }                                                                                       │
│   31 }                                                                                           │
│   32                                                                                             │
│ ❱ 33 res = aos_client.search(index='gte-multi-ft', size=20, body=query, request_timeout=55)      │
│   34 query_result=[]                                                                             │
│   35 for hit in res['hits']['hits']:                                                             │
│   36 │   row=[hit['_id'],hit['_score'],hit['_source']['title_en'], hit['_source']['organisati    │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/opensearchpy/client/utils.py: │
│ 176 in _wrapped                                                                                  │
│                                                                                                  │
│   173 │   │   │   │   │   if v is not None:                                                      │
│   174 │   │   │   │   │   │   params[p] = _escape(v)                                             │
│   175 │   │   │                                                                                  │
│ ❱ 176 │   │   │   return func(*args, params=params, headers=headers, **kwargs)                   │
│   177 │   │                                                                                      │
│   178 │   │   return _wrapped                                                                    │
│   179                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/opensearchpy/client/__init__. │
│ py:2440 in search                                                                                │
│                                                                                                  │
│   2437 │   │   if "from_" in params:                                                             │
│   2438 │   │   │   params["from"] = params.pop("from_")                                          │
│   2439 │   │                                                                                     │
│ ❱ 2440 │   │   return self.transport.perform_request(                                            │
│   2441 │   │   │   "POST",                                                                       │
│   2442 │   │   │   _make_path(index, "_search"),                                                 │
│   2443 │   │   │   params=params,                                                                │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/opensearchpy/transport.py:457 │
│ in perform_request                                                                               │
│                                                                                                  │
│   454 │   │   │   │   │   if attempt == self.max_retries:                                        │
│   455 │   │   │   │   │   │   raise e                                                            │
│   456 │   │   │   │   else:                                                                      │
│ ❱ 457 │   │   │   │   │   raise e                          

In [53]:
payload = {"inputs": "wildfires Canada"}
endpoint_name ='gte-multi-ft-se'
vector = invoke_sagemaker_endpoint_ft(endpoint_name, payload)
query={
    "size": 20,
    "query": {
        "knn": {
            "vector":{
                "vector":vector,
                "k":20
            }
        }
    }
}

res = aos_client.search(index='gte-multi-ft', size=20, body=query, request_timeout=55)
query_result=[]
for hit in res['hits']['hits']:
    row=[hit['_id'],hit['_score'],hit['_source']['title'],hit['_source']['id']]
    query_result.append(row)
query_result_df = pd.DataFrame(data=query_result,columns=["_id","relevancy_score","title",'uuid'])
display(query_result_df)

2026-07-03 13:13:12,825 - INFO - POST https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft/_search?size=20 [status:200 request:0.096s]


,_id,relevancy_score,title,uuid
0,mpHsJ58B7EZDBsAegNgv,0.929031,Fire hydrants,1fdb895a-d8ea-4b5f-af2f-e060e48af530
1,9pHtJ58B7EZDBsAenegf,0.928916,Wildfire Ignition Density,3f0a6405-0a1a-420b-ae99-068a4aeb3b95
2,KJHtJ58B7EZDBsAebeYr,0.928603,Fire hydrant,c5dc2e13-78c5-4eff-9361-a52b003bd69e
3,DJHtJ58B7EZDBsAea-Zc,0.927367,Barracks,772cb76d-54db-4dab-b091-33e48d6173f8
4,WZHtJ58B7EZDBsAecOZA,0.927215,High tides December 2010: Wave break,39bdcc75-dbaf-424d-9dbd-265c282f14f5
5,CpHtJ58B7EZDBsAea-Y8,0.926698,Fire hydrants,f647f5ed-a8f3-4a47-8ceb-977cbf090675
6,PJHsJ58B7EZDBsAei9lN,0.926068,Canada's National Earthquake Scenario Catalogu...,53183b8c-cd09-4f3d-bcbc-b023dea3990c
7,_ZHtJ58B7EZDBsAeFuCb,0.925145,Standard fire hydrant v1,33fdc525-5092-4483-9f0b-b91482cbfce9
8,WJHsJ58B7EZDBsAeVdaB,0.924761,Fire hydrants,e7e90f77-6a1b-4b9e-bc3f-650f8fd30ab8
9,b5HtJ58B7EZDBsAeceau,0.924361,Fire stations,9efd70e6-157a-43c8-861e-b6faecf6a33f


In [76]:
mapping = aos_client.indices.get_mapping(index="gte-multi-ft")

fields = mapping["gte-multi-ft"]["mappings"]["properties"]

for field, definition in fields.items():
    print(field, definition.get("type"))


2026-07-13 15:54:11,360 - INFO - GET https://search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com:443/gte-multi-ft/_mapping [status:200 request:0.009s]
contact None
coordinates geo_shape
created date
description text
eoCollection text
eoFilters None
graphicOverview None
id text
keywords text
language text
options None
organisation text
popularity long
published date
spatialRepresentation text
systemName text
temporalExtent None
title text
topicCategory text
type text
vector knn_vector


In [58]:
payload = {"inputs": "wildfires Canada"}
endpoint_name ='semantic-search-pretrain-all-MiniLM-L6-v2-1781031216'
vector = invoke_sagemaker_endpoint_ft(endpoint_name, payload)
print(vector)


Error invoking SageMaker endpoint semantic-search-pretrain-all-MiniLM-L6-v2-1781031216: An error occurred (ModelError) when calling the InvokeEndpoint operation: Received server error (500) from primary and could not load the entire response body. See https://ca-central-1.console.aws.amazon.com/cloudwatch/home?region=ca-central-1#logEventViewer:group=/aws/sagemaker/Endpoints/semantic-search-pretrain-all-MiniLM-L6-v2-1781031216 in account 759472643633 for more information.
None


In [ ]:
query={
    "size": 20,
    "query": {
        "knn": {
            "vector":{
                "vector":vector,
                "k":20
            }
        }
    }
}

res = aos_client.search(index='minilm-pretrain-knn', size=20, body=query, request_timeout=55)
query_result=[]
for hit in res['hits']['hits']:
    row=[hit['_id'],hit['_score'],hit['_source']['title'],hit['_source']['id']]
    query_result.append(row)
query_result_df = pd.DataFrame(data=query_result,columns=["_id","relevancy_score","title",'uuid'])
display(query_result_df)